# Fine-Tuning LoRA de MiniLM-L6-v2 — CIC-IDS-2017

> Wang, W. et al. (2020). *MiniLM: Deep Self-Attention Distillation for
> Task-Agnostic Compression of Pre-Trained Transformers*. arXiv:2002.10957.
>
> Checkpoint utilisé : `sentence-transformers/all-MiniLM-L6-v2`
> (6 couches, dimension cachée 384, environ 22,7 M de paramètres).

---
## ÉTAPE 0 — Installation des dépendances

In [ ]:
import subprocess, sys, importlib

print("📦 1/2 — Transformers + PEFT compatibles MiniLM...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6',
    'peft==0.18.1',
    'accelerate>=1.2.0',
    'datasets>=2.18.0',
])
print("✅ Transformers + PEFT OK")

print("\n📦 2/2 — Dépendances ML (pipeline inchangé)...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'imbalanced-learn>=0.12.0', 'scikit-learn>=1.4.0',
    'seaborn>=0.13.0', 'matplotlib>=3.8.0', 'joblib>=1.3.0',
    'xgboost>=2.0.0',
])
print("✅ Dépendances ML OK")

print("\n🔍 Vérification versions :")
for pkg in ['torch', 'transformers', 'peft', 'sklearn', 'xgboost']:
    try:
        m = importlib.import_module(pkg)
        print(f"   ✅ {pkg:18s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"   ❌ {pkg:18s} {e}")

print("\n⚠️ Redémarrez l'environnement Colab après cette cellule si les bibliothèques")
print("   étaient déjà importées dans la session.")

---
##  ÉTAPE 1 — Imports & Reproductibilité (Seed = 42)




In [ ]:
import os, random, warnings, gc, time, tracemalloc
import numpy as np
import torch
import transformers
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
import operator
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); transformers.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"🎲 Seed global = {SEED} fixé.")

import psutil
if torch.cuda.is_available():
    print(f"🖥️  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"📊 VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Go")
else:
    print("⚠️  Pas de GPU détecté.")
print(f"💾 RAM disponible : {psutil.virtual_memory().available/1e9:.1f} Go")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive monté.")
except ImportError:
    print("ℹ️  Hors Colab — Drive non monté.")


---
##  ÉTAPE 2 — Paramètres Configurables


> Hu, E.J. et al. (2022). *LoRA: Low-Rank Adaptation of Large Language Models*.
> ICLR 2022. arXiv:2106.09685


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES — modifier ici uniquement
# ═════════════════════════════════════════════════════════════════════════════
DATA_DIR               = '/content/drive/MyDrive/Dataset/CIC-IDS-2017/'
OUTPUT_DIR             = '/content/drive/MyDrive/these_llm/MiniLM_L6_CICIDS2017/'
MODEL_NAME             = 'sentence-transformers/all-MiniLM-L6-v2'  # ≈22.7M paramètres

# ── Cascade 4 niveaux ────────────────────────────────────────────────────────
MIN_SAMPLES_KEEP       = 200
ROS_THRESHOLD          = 500
MAX_SAMPLES_PER_CLASS  = 2000

# ── Features & Tokenisation ──────────────────────────────────────────────────
TOP_N_FEATURES         = 20
MAX_SEQ_LENGTH         = 256

# ── LoRA — projections d'attention BERT/MiniLM ───────────────────────────
# MiniLM expose séparément les projections Query, Key et Value.
LORA_R                 = 32
LORA_ALPHA             = 64
LORA_TARGET_MODULES    = ['query', 'value']
LORA_MODULES_TO_SAVE   = ['classifier']

# ── Entraînement ─────────────────────────────────────────────────────────────
NUM_EPOCHS             = 5
LEARNING_RATE          = 2e-4
BATCH_TRAIN            = 16
BATCH_EVAL             = 32
GRAD_ACCUM             = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("📋 Paramètres MiniLM-L6-v2 / CIC-IDS-2017 :")
print(f"   Modèle  : {MODEL_NAME}")
print(f"   Cascade : SUPP<{MIN_SAMPLES_KEEP} | ROS {MIN_SAMPLES_KEEP}-{ROS_THRESHOLD} | CONS | SUB>{MAX_SAMPLES_PER_CLASS}")
print(f"   LoRA    : r={LORA_R}, α={LORA_ALPHA}, modules={LORA_TARGET_MODULES}")
print(f"   Tête    : modules sauvegardés={LORA_MODULES_TO_SAVE}")
print(f"   Train   : {NUM_EPOCHS}ep, lr={LEARNING_RATE}, batch={BATCH_TRAIN}×{GRAD_ACCUM}={BATCH_TRAIN*GRAD_ACCUM}")


---
##  ÉTAPE 3 — Chargement & Audit du Dataset CIC-IDS-2017




In [ ]:
import os
import glob
import pandas as pd

parquet_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))
print(f"📄 {len(parquet_files)} fichier(s) Parquet trouvé(s).")
assert len(parquet_files) > 0, f"❌ Aucun fichier Parquet dans {DATA_DIR}"

dfs = []
for f in parquet_files:
    df_tmp = pd.read_parquet(f)
    dfs.append(df_tmp)
    print(f"   ✔ {os.path.basename(f)} — {df_tmp.shape[0]:,} lignes")

df = pd.concat(dfs, ignore_index=True)
print(f"\n✅ Fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r'[\s/]+', '_', regex=True)
              .str.replace(r'[^a-z0-9_]', '', regex=True)
)

LABEL_COL = "label"
if LABEL_COL not in df.columns:
    candidates = [c for c in df.columns if "label" in c or "class" in c]
    if candidates:
        LABEL_COL = candidates[0]
    else:
        raise ValueError("Aucune colonne label trouvée.")

print(f"🏷️ Colonne label : '{LABEL_COL}'")

---
##  ÉTAPE 4 — Nettoyage (doublons · Inf · NaN · clipping · LABEL_MAP)




In [ ]:
df_clean = df.copy()

n_dup = len(df_clean) - len(df_clean.drop_duplicates())
df_clean.drop_duplicates(inplace=True)
print(f"🗑️  Doublons supprimés : {n_dup:,}")

num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
n_inf = np.isinf(df_clean[num_cols]).sum().sum()
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)
n_nan = df_clean[num_cols].isnull().sum().sum()
for col in num_cols:
    if df_clean[col].isnull().any():
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
print(f"♾️  Inf→NaN : {n_inf:,} | NaN imputés : {n_nan:,}")

clip_cols = [c for c in num_cols if any(kw in c for kw in
             ['duration','bytes','packets','length','size','count','rate','iat'])]
df_clean[clip_cols] = df_clean[clip_cols].clip(lower=0)

LABEL_MAP = {
    'benign':'BENIGN','BENIGN':'BENIGN',
    'dos hulk':'DoS_Hulk','DoS Hulk':'DoS_Hulk','dos_hulk':'DoS_Hulk',
    'dos goldeneye':'DoS_GoldenEye','DoS GoldenEye':'DoS_GoldenEye',
    'dos slowloris':'DoS_Slowloris','DoS slowloris':'DoS_Slowloris','dos_slowloris':'DoS_Slowloris',
    'dos slowhttptest':'DoS_Slowhttptest','DoS Slowhttptest':'DoS_Slowhttptest',
    'ddos':'DDoS','DDoS':'DDoS','DDOS':'DDoS',
    'ftp-patator':'FTP-Patator','FTP-Patator':'FTP-Patator','ftp patator':'FTP-Patator',
    'ssh-patator':'SSH-Patator','SSH-Patator':'SSH-Patator','ssh patator':'SSH-Patator',
    'portscan':'PortScan','PortScan':'PortScan','port scan':'PortScan',
    'bot':'Bot','Bot':'Bot',
    'infiltration':'Infiltration','Infiltration':'Infiltration',
    'web attack  brute force':'Web_Attack_Brute_Force',
    'web attack - brute force':'Web_Attack_Brute_Force',
    'Web Attack  Brute Force':'Web_Attack_Brute_Force',
    'Web Attack - Brute Force':'Web_Attack_Brute_Force',
    'Web Attack Brute Force':'Web_Attack_Brute_Force',
    'web attack  xss':'Web_Attack_XSS','web attack - xss':'Web_Attack_XSS',
    'Web Attack  XSS':'Web_Attack_XSS','Web Attack XSS':'Web_Attack_XSS',
    'web attack  sql injection':'Web_Attack_Sql_Injection',
    'web attack - sql injection':'Web_Attack_Sql_Injection',
    'Web Attack  Sql Injection':'Web_Attack_Sql_Injection',
    'Web Attack Sql Injection':'Web_Attack_Sql_Injection',
    'heartbleed':'Heartbleed','Heartbleed':'Heartbleed',
}
labels_before = df_clean[LABEL_COL].str.strip().copy()
df_clean[LABEL_COL] = labels_before.map(LABEL_MAP)
n_unmapped = df_clean[LABEL_COL].isnull().sum()
if n_unmapped > 0:
    df_clean.dropna(subset=[LABEL_COL], inplace=True)
    print(f"⚠️  {n_unmapped:,} lignes non mappées supprimées.")
print(f"✅ Nettoyage terminé : {len(df_clean):,} lignes | {df_clean[LABEL_COL].nunique()} classes")


---
## 🔍 ÉTAPE 5 — Sélection des Features (Random Forest Gini, Top-20)


> Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5–32.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
import joblib

feature_cols = [c for c in df_clean.columns if c != LABEL_COL]
num_fc = df_clean[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
X_sel  = df_clean[num_fc].values
le_tmp = LabelEncoder()
y_sel  = le_tmp.fit_transform(df_clean[LABEL_COL].values)

vt = VarianceThreshold(0.0)
vt.fit(X_sel)
X_sel = X_sel[:, vt.get_support()]
num_fc = [c for c, k in zip(num_fc, vt.get_support()) if k]
print(f"📊 Features après filtre variance : {len(num_fc)}")

n_sub = min(200_000, len(X_sel))
idx_sub = np.random.choice(len(X_sel), n_sub, replace=False)
rf_sel = RandomForestClassifier(n_estimators=100, max_depth=10,
                                 n_jobs=-1, random_state=SEED)
rf_sel.fit(X_sel[idx_sub], y_sel[idx_sub])

feat_imp = pd.DataFrame({'feature': num_fc,
                          'importance': rf_sel.feature_importances_}
                        ).sort_values('importance', ascending=False)

SELECTED_FEATURES = feat_imp['feature'].head(TOP_N_FEATURES).tolist()
cum_imp = feat_imp['importance'].head(TOP_N_FEATURES).sum() / feat_imp['importance'].sum() * 100
print(f"✅ {TOP_N_FEATURES} features → {cum_imp:.1f}% de l'importance cumulée")

fig, ax = plt.subplots(figsize=(10, 6))
top20 = feat_imp.head(TOP_N_FEATURES)
ax.barh(top20['feature'][::-1], top20['importance'][::-1],
        color=sns.color_palette('coolwarm_r', TOP_N_FEATURES))
ax.set_title(f'Top-{TOP_N_FEATURES} Features — RF Gini | CIC-IDS-2017',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'features_importances.png'), dpi=150, bbox_inches='tight')
plt.show()
joblib.dump(SELECTED_FEATURES, os.path.join(OUTPUT_DIR, 'top_features.joblib'))
print(f"✅ Features sauvegardées.")


---
##  ÉTAPE 6 — Stratégie Cascade 4 Niveaux



In [ ]:
df_model = df_clean[SELECTED_FEATURES + [LABEL_COL]].copy()
counts_all = df_model[LABEL_COL].value_counts().to_dict()

print("📊 Distribution AVANT cascade :")
for cls, cnt in sorted(counts_all.items(), key=lambda x: -x[1]):
    print(f"   {cls:<35s} {cnt:>7,}")

CLASSES_SUPPRIMEES = []
sampled_dfs = []
recap = []

for cls, cnt in counts_all.items():
    df_cls = df_model[df_model[LABEL_COL] == cls]
    if cnt < MIN_SAMPLES_KEEP:
        CLASSES_SUPPRIMEES.append(cls)
        recap.append({'classe': cls, 'n_avant': cnt, 'n_après': 0,
                      'action': 'SUPPRIMÉE', 'couleur': '#E74C3C'})
        continue
    elif cnt <= ROS_THRESHOLD:
        df_cls_sampled = df_cls.copy()
        recap.append({'classe': cls, 'n_avant': cnt, 'n_après': ROS_THRESHOLD,
                      'action': f'ROS→{ROS_THRESHOLD}', 'couleur': '#3498DB'})
    elif cnt <= MAX_SAMPLES_PER_CLASS:
        df_cls_sampled = df_cls.copy()
        recap.append({'classe': cls, 'n_avant': cnt, 'n_après': cnt,
                      'action': 'CONSERVÉE', 'couleur': '#2ECC71'})
    else:
        df_cls_sampled = df_cls.sample(n=MAX_SAMPLES_PER_CLASS, random_state=SEED)
        recap.append({'classe': cls, 'n_avant': cnt,
                      'n_après': MAX_SAMPLES_PER_CLASS,
                      'action': f'SUB→{MAX_SAMPLES_PER_CLASS}', 'couleur': '#F39C12'})
    sampled_dfs.append(df_cls_sampled)
    print(f"   {cls:<35s} {cnt:>7,} → {recap[-1]['action']}")

df_bal = pd.concat(sampled_dfs, ignore_index=True).sample(frac=1, random_state=SEED)
counts_before_bal = df_bal[LABEL_COL].value_counts().to_dict()
ALL_CLASSES = sorted(counts_before_bal.keys())

print(f"\n✅ Classes SUPPRIMÉES ({len(CLASSES_SUPPRIMEES)}) : {CLASSES_SUPPRIMEES}")
print(f"✅ Classes RETENUES  ({len(ALL_CLASSES)}) : {ALL_CLASSES}")
print(f"✅ Dataset après cascade : {len(df_bal):,} samples")

df_recap = pd.DataFrame(recap)
display(df_recap[['classe', 'n_avant', 'n_après', 'action']])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts_bv = pd.Series(counts_all).sort_values(ascending=False)
colors_bv = []
for cls in counts_bv.index:
    if cls in CLASSES_SUPPRIMEES: colors_bv.append('#E74C3C')
    elif counts_all[cls] > MAX_SAMPLES_PER_CLASS: colors_bv.append('#F39C12')
    elif counts_all[cls] <= ROS_THRESHOLD: colors_bv.append('#3498DB')
    else: colors_bv.append('#2ECC71')
axes[0].barh(counts_bv.index, counts_bv.values, color=colors_bv)
for thr, col, lbl in [
    (MIN_SAMPLES_KEEP,'#E74C3C',f'Seuil supp. ({MIN_SAMPLES_KEEP})'),
    (ROS_THRESHOLD,'#3498DB',f'Seuil ROS ({ROS_THRESHOLD})'),
    (MAX_SAMPLES_PER_CLASS,'#F39C12',f'Plafond ({MAX_SAMPLES_PER_CLASS})')]:
    axes[0].axvline(thr, color=col, linestyle='--', linewidth=1.5, label=lbl)
axes[0].set_title('Distribution AVANT cascade', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nombre de samples')
axes[0].legend(fontsize=8)

counts_av = df_bal[LABEL_COL].value_counts().sort_values(ascending=False)
axes[1].barh(counts_av.index, counts_av.values,
             color=sns.color_palette('Set2', len(counts_av)))
for bar, cnt in zip(axes[1].patches, counts_av.values):
    axes[1].text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
                 f'{cnt:,}', va='center', fontsize=9)
axes[1].set_title('Distribution APRÈS cascade', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nombre de samples')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E74C3C',label='SUPPRIMÉE'),
    Patch(facecolor='#3498DB',label='ROS (post-split)'),
    Patch(facecolor='#2ECC71',label='CONSERVÉE'),
    Patch(facecolor='#F39C12',label='SOUS-ÉCHANTILLONNÉE'),
]
axes[0].legend(handles=legend_elements + axes[0].get_legend().legend_handles,
               fontsize=7.5, loc='lower right')
plt.suptitle('Cascade 4 Niveaux — CIC-IDS-2017', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'cascade_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


---
##  ÉTAPE 7 — Encodage · Split · ROS (train uniquement) · Normalisation




In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
import joblib

X_all     = df_bal[SELECTED_FEATURES].values.astype(np.float32)
y_all_str = df_bal[LABEL_COL].values

le = LabelEncoder()
y_all = le.fit_transform(y_all_str)
LABEL_ENCODER_MAP     = {i: lbl for i, lbl in enumerate(le.classes_)}
LABEL_ENCODER_MAP_INV = {lbl: i for i, lbl in LABEL_ENCODER_MAP.items()}
N_CLASSES             = len(le.classes_)

print("🏷️  LABEL_ENCODER_MAP :")
for k, v in LABEL_ENCODER_MAP.items():
    print(f"   {k:2d} → {v}")
joblib.dump(le, os.path.join(OUTPUT_DIR, 'label_encoder.joblib'))

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f"\n📊 Split AVANT ROS : train={len(X_train):,} | val={len(X_val):,} | test={len(X_test):,}")

ros_strategy = {}
for idx, cls in LABEL_ENCODER_MAP.items():
    current_count = int((y_train == idx).sum())
    if 0 < current_count < ROS_THRESHOLD:
        ros_strategy[idx] = ROS_THRESHOLD
if ros_strategy:
    ros = RandomOverSampler(sampling_strategy=ros_strategy, random_state=SEED)
    X_train, y_train = ros.fit_resample(X_train, y_train)
    print(f"🔁 ROS appliqué — Train : {len(X_train):,}")
else:
    print("ℹ️  Aucune classe dans l'intervalle ROS.")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.joblib'))
np.save(os.path.join(OUTPUT_DIR, 'X_test.npy'), X_test_sc)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)

class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_arr = np.clip(class_weights_arr, None, 10.0)
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights_arr, dtype=torch.float32)
print(f"\n⚖️  Poids de classes calculés. N classes : {N_CLASSES}")


---
## ÉTAPE 8 — Tokenisation pour MiniLM-L6-v2 (SequenceClassification)

In [ ]:
from transformers import AutoTokenizer
import datasets as hf_datasets

print(f"⏬ Chargement tokenizer {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer chargé — vocab : {tokenizer.vocab_size:,} | type : {type(tokenizer).__name__}")
print(f"   PAD id={tokenizer.pad_token_id} | SEP id={tokenizer.sep_token_id}")

# ── Prompt identique à T5 (comparabilité inter-modèles) ─────────────────────
def build_prompt(row, names):
    return 'classify: ' + ' '.join(f'{n}={v:.4f}' for n, v in zip(names, row))

exemple = build_prompt(X_train_sc[0], SELECTED_FEATURES)
n_toks = len(tokenizer(exemple, max_length=MAX_SEQ_LENGTH, truncation=True)['input_ids'])
print(f"\n📝 Exemple prompt ({n_toks} tokens) :")
print(f"   {exemple[:110]}...")

# ── Tokenisation pour SequenceClassification ─────────────────────────────────
# CLASSIFICATION ENCODEUR : labels = entiers directs — pas de tokens texte à décoder
def tokenize_dataset(X_sc, y_enc):
    texts = [build_prompt(r, SELECTED_FEATURES) for r in X_sc]
    enc   = tokenizer(texts, max_length=MAX_SEQ_LENGTH, truncation=True, padding=False)
    enc['labels'] = list(y_enc.astype(int))
    return enc

print("\n🔤 Tokenisation des 3 splits...")
hf_train = hf_datasets.Dataset.from_dict(tokenize_dataset(X_train_sc, y_train))
hf_val   = hf_datasets.Dataset.from_dict(tokenize_dataset(X_val_sc,   y_val))
hf_test  = hf_datasets.Dataset.from_dict(tokenize_dataset(X_test_sc,  y_test))
print(f"   ✅ train={len(hf_train):,} | val={len(hf_val):,} | test={len(hf_test):,}")

assert set(hf_train['labels']) == set(range(N_CLASSES)), "⚠️  Classes manquantes"
print(f"✅ Intégrité : {N_CLASSES} classes présentes dans train.")


---
## ÉTAPE 9 — MiniLM-L6-v2 + LoRA

### Vérification des modules LoRA disponibles

MiniLM repose sur l'architecture BERT et expose des projections d'attention
séparées nommées `query`, `key` et `value`. LoRA est appliqué aux projections
`query` et `value`, tandis que la tête `classifier` demeure entraînable et est
sauvegardée avec l'adaptateur.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch.nn as nn

print(f"⏬ Chargement de {MODEL_NAME}...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_CLASSES,
    ignore_mismatched_sizes=True,
)
n_params_total = sum(p.numel() for p in base_model.parameters())
print(f"✅ Modèle chargé : {n_params_total:,} paramètres ({n_params_total/1e6:.1f}M)")

# Vérification robuste des couches linéaires disponibles pour LoRA
linear_modules = sorted(set(
    name.split('.')[-1]
    for name, module in base_model.named_modules()
    if isinstance(module, nn.Linear)
))
print(f"\n🔍 Modules linéaires disponibles : {linear_modules}")

missing_targets = [tm for tm in LORA_TARGET_MODULES if tm not in linear_modules]
for tm in LORA_TARGET_MODULES:
    ok = '✅' if tm in linear_modules else '❌ NON TROUVÉ'
    print(f"   {tm:20s} : {ok}")

if missing_targets:
    raise ValueError(
        f"Modules LoRA MiniLM introuvables : {missing_targets}. "
        f"Modules linéaires détectés : {linear_modules}"
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=LORA_MODULES_TO_SAVE,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100 * n_trainable / n_params_total
print(f"\n🎯 MiniLM + LoRA : {n_trainable:,} paramètres entraînables ({pct:.2f}%)")
print("   Projections adaptées : query et value")
print("   Tête entraînable   : classifier")


---
## ÉTAPE 10 — Configuration & Entraînement

### Trainer standard

MiniLM utilise le `Trainer` standard car l'inférence repose sur un unique
forward pass de classification, sans décodage autorégressif.

`DataCollatorWithPadding` applique un padding dynamique aux `input_ids` et à
l'`attention_mask`; les labels restent des entiers scalaires.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, TrainerCallback
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score as sk_f1
import shutil, copy

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
)



BEST_CKPT_DIR = './best_minilm_checkpoint'

class BestModelCallback(TrainerCallback):
    """
    Évalue le val set à chaque epoch, sauvegarde le meilleur modèle selon F1-macro.

    Contourne le bug KeyError du Trainer en évaluant avec un DataLoader direct
    plutôt qu'en passant par prediction_loop() du Trainer.
    """
    def __init__(self, val_dataset, tokenizer, data_collator, n_classes,
                 best_dir, device):
        self.val_dataset   = val_dataset
        self.tokenizer     = tokenizer
        self.data_collator = data_collator
        self.n_classes     = n_classes
        self.best_dir      = best_dir
        self.device        = device
        self.best_f1       = -1.0
        self.best_epoch    = -1
        self.history       = []   # [(epoch, f1_val)]

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        """Évaluation F1-macro sur val set à la fin de chaque epoch."""
        from torch.utils.data import DataLoader as _DL

        model.eval()
        loader = _DL(
            self.val_dataset.with_format('torch'),
            batch_size=64, shuffle=False,
            collate_fn=self.data_collator
        )
        preds_all, labels_all = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(self.device)
                attn_mask = batch['attention_mask'].to(self.device)
                labels_b  = batch['labels'].cpu().numpy().tolist()
                out = model(input_ids=input_ids, attention_mask=attn_mask)
                pred = out.logits.argmax(dim=-1).cpu().numpy().tolist()
                preds_all.extend(pred)
                labels_all.extend(labels_b)

        model.train()

        f1_val = sk_f1(labels_all, preds_all, average='macro',
                       zero_division=0, labels=list(range(self.n_classes)))
        epoch  = int(state.epoch)
        self.history.append((epoch, round(float(f1_val), 4)))
        print(f"\n   📊 Epoch {epoch} — F1-macro val : {f1_val:.4f}", end="")

        if f1_val > self.best_f1:
            self.best_f1    = f1_val
            self.best_epoch = epoch
            # ── Sauvegarde du meilleur modèle ──────────────────────────────────
            #
            # STRATÉGIE : torch.save() du state_dict() COMPLET du PeftModel.
            #
            # Pourquoi cette approche est fiable :
            #   • model.state_dict() retourne TOUS les paramètres avec leurs
            #     vrais noms (ex: 'base_model.model.classifier.weight')
            #   • v.detach().cpu() copie les tenseurs SANS les attacher au
            #     graphe de calcul → l'entraînement continue normalement
            #   • Au rechargement : même architecture PeftModel →
            #     load_state_dict() mappe toutes les clés parfaitement
            #
            # NE PAS utiliser merge_and_unload() — destructif en cours d'entraînement.
            # NE PAS utiliser save_pretrained() seul → omet la tête de classif.
            os.makedirs(self.best_dir, exist_ok=True)
            full_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            torch.save(full_state,
                       os.path.join(self.best_dir, 'full_model_state.pt'))
            self.tokenizer.save_pretrained(self.best_dir)
            print(f" ← 🏆 NOUVEAU MEILLEUR (state_dict complet sauvegardé)", end="")
        print()

best_callback = BestModelCallback(
    val_dataset   = hf_val,
    tokenizer     = tokenizer,
    data_collator = data_collator,
    n_classes     = N_CLASSES,
    best_dir      = BEST_CKPT_DIR,
    device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)

training_args = TrainingArguments(
    output_dir='./output_minilm_cicids',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.06,
    fp16=True,
    eval_strategy='no',            # évaluation gérée par BestModelCallback
    save_strategy='no',            # sauvegarde gérée par BestModelCallback
    load_best_model_at_end=False,  # chargement manuel après training
    logging_steps=50,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=0,
    remove_unused_columns=True,
)

class WeightedTrainer(Trainer):
    """
    CrossEntropyLoss pondérée pour classification déséquilibrée.
    Évaluation val et sélection meilleur checkpoint via BestModelCallback.
    """
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False,
                     num_items_in_batch=None):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        cw      = self.class_weights.to(outputs.logits.device)
        loss    = nn.CrossEntropyLoss(weight=cw)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    class_weights=CLASS_WEIGHTS_TENSOR,
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=None,           # évaluation gérée par BestModelCallback
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[best_callback],   # ← évaluation + checkpoint personnalisés
)

torch.cuda.reset_peak_memory_stats()
t_start = time.time()
print(f"🚀 Entraînement MiniLM-L6-v2 | {NUM_EPOCHS} ep | lr={LEARNING_RATE}")
print(f"   Sélection meilleur checkpoint : F1-macro val (via BestModelCallback)")
train_result  = trainer.train()
t_train_min   = (time.time() - t_start) / 60
vram_peak_go  = torch.cuda.max_memory_allocated() / 1e9

print(f"\n✅ Entraînement terminé !")
print(f"   ⏱️  Temps       : {t_train_min:.1f} min")
print(f"   🖥️  VRAM peak   : {vram_peak_go:.2f} Go")
print(f"   📉  Train loss  : {train_result.training_loss:.4f}")
print(f"   🏆  Best F1-val : {best_callback.best_f1:.4f} (epoch {best_callback.best_epoch})")

# ── Chargement du meilleur checkpoint ────────────────────────────────────────
print(f"\n⏬ Rechargement du meilleur checkpoint (epoch {best_callback.best_epoch})...")
del model
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForSequenceClassification as _AMC
_base = _AMC.from_pretrained(MODEL_NAME, num_labels=N_CLASSES, ignore_mismatched_sizes=True)
from peft import LoraConfig as _LC2, get_peft_model as _gpm2, TaskType as _TT2
_lc2 = _LC2(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES,
             lora_dropout=0.05, bias='none', modules_to_save=LORA_MODULES_TO_SAVE, task_type=_TT2.SEQ_CLS)
model = _gpm2(_base, _lc2)
_sp = os.path.join(BEST_CKPT_DIR, 'full_model_state.pt')
_ss = torch.load(_sp, map_location='cpu')
_m, _u = model.load_state_dict(_ss, strict=True)
print(f"   ✅ state_dict chargé — manquant:{len(_m)} inattendu:{len(_u)}")
model.eval()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(DEVICE)
print(f"✅ Meilleur modèle rechargé depuis '{BEST_CKPT_DIR}'")

# ── Courbes d'entraînement ────────────────────────────────────────────────────
history   = trainer.state.log_history
tr_losses = [(e['step'], e['loss']) for e in history
             if 'loss' in e and 'eval_loss' not in e]
val_hist  = best_callback.history   # [(epoch, f1_val)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

if tr_losses:
    steps, losses = zip(*tr_losses)
    axes[0].plot(steps, losses, color='#0072B2', linewidth=1.5)
    axes[0].set_title('Loss train — MiniLM-L6-v2\nCIC-IDS-2017',
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('CrossEntropyLoss (pondérée)')
    axes[0].grid(alpha=0.3)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

if val_hist:
    ep_v, f1_v = zip(*val_hist)
    axes[1].plot(ep_v, f1_v, 's-', color='#009E73', linewidth=2, markersize=9)
    best_ep = best_callback.best_epoch
    best_f1 = best_callback.best_f1
    axes[1].axhline(best_f1, color='#D55E00', linestyle='--', linewidth=1.5,
                    label=f'Meilleur : {best_f1:.4f} (ep {best_ep})')
    for xi, yi in zip(ep_v, f1_v):
        axes[1].annotate(f'{yi:.4f}', (xi, yi),
                         textcoords='offset points', xytext=(0, 10),
                         ha='center', fontsize=9, fontweight='bold')
    # Marquer le meilleur epoch
    axes[1].scatter([best_ep], [best_f1], color='#D55E00', s=150, zorder=5,
                    marker='*', label='Checkpoint sauvegardé')
    axes[1].set_title('F1-macro validation par epoch\n(BestModelCallback)',
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1-macro (val set)')
    axes[1].set_ylim(max(0, min(f1_v) - 0.05), 1.05)
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'courbes_entrainement.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Courbes sauvegardées.")


In [ ]:
!pip uninstall -y torch torchvision torchaudio datasets

In [ ]:
!pip install -U torch torchvision torchaudio

In [ ]:
!pip install -U transformers datasets accelerate sentencepiece

In [ ]:
import torch
import torchvision
import datasets
import transformers

print(torch.__version__)
print(torchvision.__version__)
print(datasets.__version__)
print(transformers.__version__)


---
## ÉTAPE 11 — Évaluation scientifique + **temps d'inférence**

### Inférence MiniLM : argmax direct

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score, matthews_corrcoef
)
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()
model.to(DEVICE)

loader = DataLoader(
    hf_test.with_format('torch'),
    batch_size=64,
    shuffle=False,
    collate_fn=data_collator
)

all_preds, all_labels_true = [], []
print("🔍 Inférence MiniLM — argmax direct (forward pass unique)...")
total_batches = len(loader)

# ── Mesure du temps d'inférence — chrono sur TOUT le test set ───────────────
# On mesure après synchronisation CUDA pour avoir le vrai temps GPU.
# Référence : PyTorch docs — torch.cuda.synchronize() vide la file CUDA
#             avant de lire le chrono (évite les mesures sous-estimées).
torch.cuda.synchronize()
t_infer_start = time.perf_counter()

with torch.no_grad():
    for bi, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels_batch   = batch['labels'].to(DEVICE)

        # ── Inférence MiniLM ──────────────────────────────────────────────
        # Un seul forward pass → logits (batch, N_CLASSES) → argmax

        outputs     = model(input_ids=input_ids, attention_mask=attention_mask)
        preds_batch = outputs.logits.argmax(dim=-1)

        all_preds.extend(preds_batch.cpu().numpy().tolist())
        all_labels_true.extend(labels_batch.cpu().numpy().tolist())

        if (bi+1) % 10 == 0 or (bi+1) == total_batches:
            print(f"   {bi+1}/{total_batches} ({100*(bi+1)/total_batches:.0f}%)")

torch.cuda.synchronize()
t_infer_end = time.perf_counter()

# ── Calcul des métriques de temps d'inférence ────────────────────────────────
t_infer_total_s       = t_infer_end - t_infer_start
n_test_samples        = len(all_preds)
t_infer_ms_per_sample = (t_infer_total_s / n_test_samples) * 1000
samples_per_second    = n_test_samples / t_infer_total_s

print(f"\n⏱️  TEMPS D'INFÉRENCE :")
print(f"   Total test set     : {t_infer_total_s:.2f} s ({n_test_samples:,} samples)")
print(f"   Par échantillon    : {t_infer_ms_per_sample:.3f} ms/sample")
print(f"   Débit              : {samples_per_second:.0f} samples/s")

y_pred      = np.array(all_preds)
y_true_eval = np.array(all_labels_true)
class_names  = [LABEL_ENCODER_MAP[i] for i in range(N_CLASSES)]
labels_range = list(range(N_CLASSES))

f1_macro    = f1_score(y_true_eval, y_pred, average='macro',    zero_division=0, labels=labels_range)
f1_weighted = f1_score(y_true_eval, y_pred, average='weighted', zero_division=0, labels=labels_range)
accuracy    = accuracy_score(y_true_eval, y_pred)
mcc_val     = matthews_corrcoef(y_true_eval, y_pred)
precision_m = precision_score(y_true_eval, y_pred, average='macro', zero_division=0, labels=labels_range)
recall_m    = recall_score(y_true_eval, y_pred, average='macro', zero_division=0, labels=labels_range)
recalls_pc  = recall_score(y_true_eval, y_pred, average=None, zero_division=0, labels=labels_range)
gmean_val   = reduce(operator.mul, [max(r,1e-10) for r in recalls_pc], 1.0)**(1.0/N_CLASSES)

gpu_hours   = t_train_min / 60
indice_frug = f1_macro / (gpu_hours * vram_peak_go) if gpu_hours > 0 else 0

print("\n" + "="*60)
print("📊 RÉSULTATS — MiniLM-L6-v2 LoRA | CIC-IDS-2017")
print("="*60)
print(f"   F1-macro     (principale)  : {f1_macro:.4f}")
print(f"   F1-weighted               : {f1_weighted:.4f}")
print(f"   G-mean                    : {gmean_val:.4f}")
print(f"   MCC                       : {mcc_val:.4f}")
print(f"   Accuracy                  : {accuracy:.4f}")
print(f"   Précision macro           : {precision_m:.4f}")
print(f"   Rappel macro              : {recall_m:.4f}")
print(f"   ─────────────────────────────────────────────────")
print(f"   VRAM peak                 : {vram_peak_go:.2f} Go")
print(f"   Temps train               : {t_train_min:.1f} min")
print(f"   Temps inférence/sample    : {t_infer_ms_per_sample:.3f} ms")
print(f"   Débit inférence           : {samples_per_second:.0f} samples/s")
print(f"   Indice frugalité (IF_MAS) : {indice_frug:.4f}")
print("="*60)

print("\n📋 Rapport de classification :")
print(classification_report(y_true_eval, y_pred, labels=labels_range,
                             target_names=class_names, zero_division=0, digits=4))

cm      = confusion_matrix(y_true_eval, y_pred, labels=labels_range)
row_sum = cm.sum(axis=1, keepdims=True)
cm_pct  = cm.astype(float) / np.where(row_sum==0, 1, row_sum) * 100

fig, axes = plt.subplots(1, 2, figsize=(max(16, N_CLASSES*1.4), max(7, N_CLASSES*0.7)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Matrice confusion — absolue', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Matrice confusion — % par ligne', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'matrices_confusion.png'), dpi=150, bbox_inches='tight')
plt.show()


---
##  ÉTAPE 13 — Sauvegarde & Tableau Récapitulatif Final




In [ ]:
import shutil

LOCAL_OUTPUT_DIR = './output_minilm_cicids'
ADAPTER_DIR = os.path.join(LOCAL_OUTPUT_DIR, 'lora_adapter')

if os.path.exists(ADAPTER_DIR):
    shutil.rmtree(ADAPTER_DIR)
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Conserver aussi le meilleur state_dict complet pour la cellule de reprise.
best_state_path = os.path.join(BEST_CKPT_DIR, 'full_model_state.pt')
if os.path.exists(best_state_path):
    shutil.copy2(best_state_path, os.path.join(ADAPTER_DIR, 'full_model_state.pt'))
    print("✅ Meilleur state_dict complet copié avec l'adaptateur.")

size_lora = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
) / 1e6
print(f"✅ Adaptateur LoRA : {size_lora:.1f} Mo")

n_lora   = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total  = sum(p.numel() for p in model.parameters())
pct_lora = 100 * n_lora / n_total

results_row = {
    'Modèle'                   : f'MiniLM-L6-v2 (LoRA r={LORA_R}, {NUM_EPOCHS}ep)',
    'Dataset'                  : 'CIC-IDS-2017',
    'Architecture'             : 'Encodeur + ClassificationHead (argmax)',
    'Inférence'                : 'argmax direct (1 forward pass)',
    'N_params_total'           : f'{n_total/1e6:.1f}M',
    'N_params_LoRA'            : n_lora,
    'Pct_trainable'            : f'{pct_lora:.2f}%',
    'Cascade'                  : f'Supp<{MIN_SAMPLES_KEEP}|ROS{MIN_SAMPLES_KEEP}-{ROS_THRESHOLD}|Cons|Sub>{MAX_SAMPLES_PER_CLASS}',
    'N_classes'                : N_CLASSES,
    'Classes_supprimees'       : str(CLASSES_SUPPRIMEES),
    'F1-macro'                 : round(f1_macro,    4),
    'F1-weighted'              : round(f1_weighted, 4),
    'G-mean'                   : round(gmean_val,   4),
    'MCC'                      : round(mcc_val,     4),
    'Accuracy'                 : round(accuracy,    4),
    'VRAM_peak_Go'             : round(vram_peak_go, 2),
    'Temps_train_min'          : round(t_train_min,  1),
    'GPU-hours'                : round(t_train_min/60, 4),
    'Temps_infer_ms_per_sample': round(t_infer_ms_per_sample, 3),
    'Debit_infer_samples_s'    : round(samples_per_second, 0),
    'T_infer_total_s'          : round(t_infer_total_s, 2),
    'Indice_frugalite'         : round(indice_frug, 4),
}

df_final = pd.DataFrame([results_row])
print("\n📊 Tableau récapitulatif final (incluant temps inférence) :")
display(df_final.T)

csv_path = os.path.join(OUTPUT_DIR, 'tableau_minilm_cicids.csv')
df_final.to_csv(csv_path, index=False)
print(f"\n✅ CSV : {csv_path}")

try:
    dest = os.path.join(OUTPUT_DIR, 'output_minilm_cicids')
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(LOCAL_OUTPUT_DIR, dest)
    print(f"✅ Copié vers Drive : {dest}")
except Exception as e:
    print(f"⚠️  Copie Drive : {e}")

print("\n" + "="*60)
print("🏁 PIPELINE MiniLM-L6-v2 CIC-IDS-2017 TERMINÉ")
print("="*60)
print(f"   F1-macro               : {f1_macro:.4f}")
print(f"   MCC                    : {mcc_val:.4f}")
print(f"   G-mean                 : {gmean_val:.4f}")
print(f"   VRAM peak              : {vram_peak_go:.2f} Go")
print(f"   Temps train            : {t_train_min:.1f} min")
print(f"   Temps inférence/sample : {t_infer_ms_per_sample:.3f} ms")
print(f"   Débit inférence        : {samples_per_second:.0f} samples/s")
print(f"   Indice frugalité       : {indice_frug:.4f}")
print(f"   Classes supprimées     : {CLASSES_SUPPRIMEES}")
print("="*60)


---
## 🔄 CELLULE REPRISE — Crash Recovery MiniLM/CIC-IDS-2017

In [ ]:
import os, subprocess, sys

OUTPUT_DIR_R = '/content/drive/MyDrive/these_llm/MiniLM_L6_CICIDS2017/'
MODEL_NAME_R = 'sentence-transformers/all-MiniLM-L6-v2'

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.18.1',
    'accelerate>=1.2.0', 'datasets>=2.18.0'
])

import numpy as np, torch, joblib, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

ADAPTER_PATH = os.path.join(
    OUTPUT_DIR_R, 'output_minilm_cicids', 'lora_adapter'
)
tok_r = AutoTokenizer.from_pretrained(ADAPTER_PATH)

le_r_tmp = joblib.load(os.path.join(OUTPUT_DIR_R, 'label_encoder.joblib'))
N_C_R = len(le_r_tmp.classes_)

# Reconstruire exactement l'architecture MiniLM + LoRA utilisée à l'entraînement.
_base_rep = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_R,
    num_labels=N_C_R,
    ignore_mismatched_sizes=True,
)
_lc_r = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['query', 'value'],
    modules_to_save=['classifier'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)
model_r = get_peft_model(_base_rep, _lc_r)

_sp_r = os.path.join(ADAPTER_PATH, 'full_model_state.pt')
if not os.path.exists(_sp_r):
    raise FileNotFoundError(
        f"full_model_state.pt non trouvé dans {ADAPTER_PATH}. "
        "Exécutez d'abord la cellule de sauvegarde finale."
    )

_ss_r = torch.load(_sp_r, map_location='cpu')
missing_r, unexpected_r = model_r.load_state_dict(_ss_r, strict=True)
print(f"✅ state_dict complet rechargé — manquant:{len(missing_r)} inattendu:{len(unexpected_r)}")

model_r.eval()
DEV_R = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_r.to(DEV_R)

sc_r = joblib.load(os.path.join(OUTPUT_DIR_R, 'scaler.joblib'))
sf_r = joblib.load(os.path.join(OUTPUT_DIR_R, 'top_features.joblib'))
X_ts = np.load(os.path.join(OUTPUT_DIR_R, 'X_test.npy'))
y_ts = np.load(os.path.join(OUTPUT_DIR_R, 'y_test.npy'))
LEM_R = {i: lbl for i, lbl in enumerate(le_r_tmp.classes_)}

print(f"✅ MiniLM-L6-v2 rechargé ({N_C_R} classes).")
print("   Inférence : model(input_ids, attention_mask).logits.argmax(-1)")